# 12 · HyDE (Hypothetical Document Embeddings)

Embed a guessed answer, then search near that vector.

**Analogy handbook:** [hyde](../retriever-analogy-handbook.html#hyde)  
**Prerequisite:** run `00_basics_concepts.ipynb` once (or the setup cells below) so the Chroma index exists.

### Learning loop
1. Skim the analogy for this technique  
2. Run setup (reuse index if possible)  
3. Run the practical cells  
4. Ask: *Did this fix the failure mode, or only reshuffle noise?*


## Shared setup

These cells install packages, load the Llama 2 PDF, build/load the Chroma index, and define helpers.

> Prefer `REBUILD_INDEX = False` after the first successful build so later method notebooks reuse the same store.


### Learning: !pip install langchain_community langchain_text_splitters langchain_op

**What you'll learn:** Install the packages this notebook needs.

**What this cell does:** Installs required Python packages into the runtime.

**Watch for:** Run once; restart runtime if Colab asks.



In [ ]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 2.2 MB/s eta 0:00:00


### Learning: IMPORTS

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `IMPORTS` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
print("All imports and setup starting...")

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import getpass
import os
import shutil

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_classic.chains.hyde.base import (
    HypotheticalDocumentEmbedder
)

All imports and setup starting...


/tmp/ipykernel_520/2286258816.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Learning: from google.colab import userdata

**What you'll learn:** Bring in LangChain, embeddings, and vector-store modules.

**What this cell does:** Runs `from google.colab import userdata` and prints intermediate results you can inspect.

**Watch for:** If an import fails, re-run the install cell.



In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

### Learning: OPENAI API KEY

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `OPENAI API KEY` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 2. OPENAI API KEY
# ============================================================

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass(
        "Enter your OpenAI API key: "
    )

print("OpenAI API key configured successfully.")

OpenAI API key configured successfully.


### Learning: DATA DIRECTORY

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `DATA DIRECTORY` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 3. DATA DIRECTORY
# ============================================================

DATA_DIR = Path(
    r"/content/"
)

preferred_pdf = DATA_DIR / "llama2-research-paper.pdf"

### Learning: FIND PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `FIND PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [ ]:
# ============================================================
# 4. FIND PDF
# ============================================================

if preferred_pdf.exists():

    PDF_PATH = preferred_pdf

else:

    available_pdfs = sorted(
        DATA_DIR.glob("*.pdf")
    )

    if len(available_pdfs) == 1:

        PDF_PATH = available_pdfs[0]

    elif len(available_pdfs) == 0:

        raise FileNotFoundError(
            f"No PDF file was found inside:\n{DATA_DIR}"
        )

    else:

        raise RuntimeError(
            "Multiple PDF files were found. "
            "Please set PDF_PATH manually.\n"
            + "\n".join(
                str(path)
                for path in available_pdfs
            )
        )


print("PDF found:")
print(PDF_PATH)

PDF found:
/content/llama2-research-paper.pdf


### Learning: LOAD PDF

**What you'll learn:** Locate and load the Llama 2 paper as Document pages.

**What this cell does:** Runs `LOAD PDF` and prints intermediate results you can inspect.

**Watch for:** Confirm page count and first-page text look sane.



In [15]:
# ============================================================
# 5. LOAD PDF
# ============================================================

loader = PyPDFLoader(
    str(PDF_PATH)
)

pages = loader.load()

print(
    f"\nTotal PDF pages loaded: {len(pages)}"
)


# ============================================================
# 6. INSPECT FIRST PAGE
# ============================================================

print("\nFirst-page metadata:")
print(
    pages[0].metadata
)

print("\nFirst 1,000 characters:")
print(
    pages[0].page_content[:1000]
)


Total PDF pages loaded: 77

First-page metadata:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}

First 1,000 characters:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Art

### Learning: IDENTIFY PAPER SECTIONS

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Defines helper logic for: IDENTIFY PAPER SECTIONS.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 7. IDENTIFY PAPER SECTIONS
# ============================================================

def identify_section(
    paper_page: int
) -> str:

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


# ============================================================
# 8. ADD METADATA
# ============================================================

for page_document in pages:

    page_index = int(
        page_document.metadata.get(
            "page",
            0
        )
    )

    paper_page = (
        page_index + 1
    )

    page_document.metadata.update(
        {
            "paper": "Llama 2",
            "organization": "Meta",
            "year": 2023,
            "document_type": "research_paper",
            "paper_page": paper_page,
            "section": identify_section(
                paper_page
            ),
            "access_level": "public",
        }
    )


print("\nMetadata after enrichment:")

for page_document in pages[:5]:

    print(
        page_document.metadata
    )



Metadata after enrichment:
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\complete_content_new\\Full-Stack-GenAI-Bootcamp-1.0\\Class-37-08-Aug-2026-prompting\\data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public'}
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',

### Learning: TEXT SPLITTING

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `TEXT SPLITTING` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 9. TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

chunks = text_splitter.split_documents(
    pages
)

print(
    f"\nTotal pages: {len(pages)}"
)

print(
    f"Total chunks: {len(chunks)}"
)


# ============================================================
# 10. ADD CHUNK IDs
# ============================================================

for chunk_number, chunk in enumerate(
    chunks
):

    paper_page = chunk.metadata.get(
        "paper_page",
        "unknown"
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"llama2-page-"
        f"{paper_page}-"
        f"chunk-{chunk_number}"
    )


print("\nFirst chunk content:")

print(
    chunks[0].page_content[:1000]
)

print("\nFirst chunk metadata:")

print(
    chunks[0].metadata
)


Total pages: 77
Total chunks: 343

First chunk content:
Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Shar

### Learning: CREATE EMBEDDING MODEL

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE EMBEDDING MODEL` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 11. CREATE EMBEDDING MODEL
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 12. TEST EMBEDDING MODEL
# ============================================================

test_vector = embeddings.embed_query(
    "What is Llama 2?"
)

print(
    f"\nEmbedding dimensions: "
    f"{len(test_vector)}"
)

print(
    f"First 10 values: "
    f"{test_vector[:10]}"
)



Embedding dimensions: 1536
First 10 values: [0.0027942657470703125, -0.0521240234375, -0.021087646484375, -0.055419921875, -0.026397705078125, 0.028961181640625, -0.002071380615234375, 0.034759521484375, -0.0164794921875, -0.0245208740234375]


### Learning: CHROMA CONFIGURATION

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CHROMA CONFIGURATION` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 13. CHROMA CONFIGURATION
# ============================================================

PERSIST_DIRECTORY = (
    DATA_DIR
    / "chroma_llama2_retriever"
)

COLLECTION_NAME = (
    "llama2_retriever_demo"
)


### Learning: CREATE OR LOAD VECTOR STORE

**What you'll learn:** Build or reload the vector index used by retrievers.

**What this cell does:** Runs `CREATE OR LOAD VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Use REBUILD_INDEX=False after the first successful build.



In [ ]:
# ============================================================
# 14. CREATE OR LOAD VECTOR STORE
# ============================================================

# True  = rebuild complete vector DB
# False = reuse existing vector DB

REBUILD_INDEX = True

### Learning: VERIFY VECTOR STORE

**What you'll learn:** Break pages into retrieval-sized chunks.

**What this cell does:** Runs `VERIFY VECTOR STORE` and prints intermediate results you can inspect.

**Watch for:** Chunk size trades precision vs context — inspect a sample.



In [ ]:
if REBUILD_INDEX:

    print(
        "\nRebuilding vector store..."
    )

    if PERSIST_DIRECTORY.exists():

        shutil.rmtree(
            PERSIST_DIRECTORY,
            ignore_errors=True
        )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(
            PERSIST_DIRECTORY
        ),
        collection_configuration={
            "hnsw": {
                "space": "cosine"
            }
        },
    )

    print(
        "New vector store created."
    )

else:

    if not PERSIST_DIRECTORY.exists():

        print(
            "\nExisting vector DB "
            "not found."
        )

        print(
            "Creating a new vector store..."
        )

        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
            collection_configuration={
                "hnsw": {
                    "space": "cosine"
                }
            },
        )

        print(
            "New vector store created."
        )

    else:

        print(
            "\nLoading existing "
            "vector store..."
        )

        vector_store = Chroma(
            collection_name=COLLECTION_NAME,
            embedding_function=embeddings,
            persist_directory=str(
                PERSIST_DIRECTORY
            ),
        )

        print(
            "Existing vector store "
            "loaded."
        )


# ============================================================
# 15. VERIFY VECTOR STORE
# ============================================================

stored_count = (
    vector_store
    ._collection
    .count()
)

print(
    f"\nStored chunks: "
    f"{stored_count}"
)

print(
    f"Persisted at: "
    f"{PERSIST_DIRECTORY}"
)



Rebuilding vector store...
New vector store created.

Stored chunks: 343
Persisted at: D:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\Class-37-08-Aug-2026-prompting\data\chroma_llama2_retriever


### Learning: RETRIEVER + HYDE PRACTICAL

**What you'll learn:** Embed a guessed answer, then search near that vector.

**What this cell does:** Runs `RETRIEVER + HYDE PRACTICAL` and prints intermediate results you can inspect.

**Watch for:** Helps when the question vocabulary differs from the docs.



In [ ]:
# ============================================================
# RETRIEVER + HYDE PRACTICAL
# ============================================================

### Learning: NORMAL DENSE RETRIEVAL TEST

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `NORMAL DENSE RETRIEVAL TEST` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 16. NORMAL DENSE RETRIEVAL TEST
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

### Learning: normal_documents = (

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: normal_documents = (.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
normal_documents = (
    vector_store
    .similarity_search(
        query=query,
        k=4
    )
)


print(
    "\nNORMAL DENSE RETRIEVAL"
)

print(
    "=" * 80
)

for i, document in enumerate(
    normal_documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        document.page_content[:700]
    )


NORMAL DENSE RETRIEVAL

Result 1
Page: 3
Section: introduction
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Page: 3
Section: introduction
the community to advance AI alignment research.
In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and
Llama 2-Chat, at scales up to 70B parameters. On the series of helpfulness and safety benchmarks we tested,
Llama 2-Chat models generally perform better than existing open-source models. They also appear to
be on par with some of the closed-source models, at least on the human evaluations we performed (see
Figures 1 and 3). We have taken measures to increase the safety of these models, using safety-specific data
annotation and tuning, as well as conducting red-teaming

### Learning: CREATE HYDE EMBEDDINGS

**What you'll learn:** Authenticate so embedding and chat calls can run.

**What this cell does:** Runs `CREATE HYDE EMBEDDINGS` and prints intermediate results you can inspect.

**Watch for:** Never hardcode secrets in shared notebooks.



In [ ]:
# ============================================================
# 17. CREATE HYDE EMBEDDINGS
# ============================================================

hyde_embeddings = (
    HypotheticalDocumentEmbedder
    .from_llm(
        llm=OpenAI(
            model="gpt-3.5-turbo-instruct",
            temperature=0
        ),
        base_embeddings=embeddings,
        prompt_key="web_search"
    )
)


print(
    "\nHyDE embedding model created."
)


HyDE embedding model created.


### Learning: HYDE FLOW

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `HYDE FLOW` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 18. HYDE FLOW
# ============================================================

"""
User Query
    ↓
LLM
    ↓
Hypothetical Document
    ↓
Embedding Model
    ↓
HyDE Query Vector
    ↓
Vector Search
    ↓
Real Documents
"""

### Learning: GENERATE HYPOTHETICAL DOCUMENT

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Executes retrieval/generation for: GENERATE HYPOTHETICAL DOCUMENT.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ============================================================
# 19. GENERATE HYPOTHETICAL DOCUMENT
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)

hypothetical_document = (
    hyde_embeddings
    .llm_chain
    .invoke(
        {
            "QUESTION": query
        }
    )
)


print(
    "\nORIGINAL QUERY:"
)

print(
    query
)

print(
    "\nHYPOTHETICAL DOCUMENT:"
)

print(
    hypothetical_document
)



ORIGINAL QUERY:
How does Llama 2 improve safety?

HYPOTHETICAL DOCUMENT:
 Llama 2 is a revolutionary safety system that has been designed to enhance safety in various settings. This innovative system utilizes advanced technology and cutting-edge features to provide a comprehensive safety solution. One of the main ways in which Llama 2 improves safety is through its real-time monitoring capabilities. The system is equipped with sensors and cameras that constantly monitor the environment and detect any potential hazards or risks. This allows for immediate action to be taken, preventing accidents or incidents from occurring. Additionally, Llama 2 has a built-in emergency response feature that can be activated in case of an emergency. This feature quickly alerts the necessary authorities and provides them with the exact location of the incident, allowing for a swift and efficient response. Furthermore, Llama 2 also has a user-friendly interface that allows individuals to easily access saf

### Learning: GENERATE HYDE VECTOR

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `GENERATE HYDE VECTOR` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:


# ============================================================
# 20. GENERATE HYDE VECTOR
# ============================================================

hyde_vector = (
    hyde_embeddings
    .embed_query(
        query
    )
)


print(
    "\nHyDE embedding dimension:",
    len(hyde_vector)
)

print(
    "\nFirst 10 embedding values:"
)

print(
    hyde_vector[:10]
)



HyDE embedding dimension: 1536

First 10 embedding values:
[0.01241302490234375, -0.01287078857421875, 0.0023708343505859375, -0.00775146484375, -0.0204620361328125, -0.002941131591796875, 0.011077880859375, 0.0184783935546875, 0.0157470703125, 0.02001953125]


### Learning: SEARCH USING HYDE VECTOR

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Executes retrieval/generation for: SEARCH USING HYDE VECTOR.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 21. SEARCH USING HYDE VECTOR
# ============================================================

hyde_documents = (
    vector_store
    .similarity_search_by_vector(
        embedding=hyde_vector,
        k=4
    )
)


print(
    "\nHYDE RETRIEVAL RESULTS"
)

print(
    "=" * 80
)


for i, document in enumerate(
    hyde_documents,
    start=1
):

    print(
        f"\nResult {i}"
    )

    print(
        "Paper page:",
        document.metadata.get(
            "paper_page"
        )
    )

    print(
        "Section:",
        document.metadata.get(
            "section"
        )
    )

    print(
        "Chunk ID:",
        document.metadata.get(
            "chunk_id"
        )
    )

    print(
        "-" * 80
    )

    print(
        document.page_content[:1000]
    )



HYDE RETRIEVAL RESULTS

Result 1
Paper page: 3
Section: introduction
Chunk ID: llama2-page-3-chunk-10
--------------------------------------------------------------------------------
continue to improve the safety of those models, paving the way for more responsible development of LLMs.
We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, such as
the emergence of tool usage and temporal organization of knowledge.
3

Result 2
Paper page: 4
Section: introduction
Chunk ID: llama2-page-4-chunk-12
--------------------------------------------------------------------------------
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in t

### Learning: COMPARE NORMAL VS HYDE

**What you'll learn:** Embed a guessed answer, then search near that vector.

**What this cell does:** Runs `COMPARE NORMAL VS HYDE` and prints intermediate results you can inspect.

**Watch for:** Helps when the question vocabulary differs from the docs.



In [ ]:
# ============================================================
# 22. COMPARE NORMAL VS HYDE
# ============================================================

query = (
    "How does Llama 2 improve safety?"
)


### Learning: Normal Dense Retrieval

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Executes retrieval/generation for: Normal Dense Retrieval.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ------------------------------------------------------------
# Normal Dense Retrieval
# ------------------------------------------------------------

normal_documents = (
    vector_store
    .similarity_search(
        query=query,
        k=4
    )
)

### Learning: HyDE Retrieval

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Runs `HyDE Retrieval` and prints intermediate results you can inspect.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
# ------------------------------------------------------------
# HyDE Retrieval
# ------------------------------------------------------------

hyde_vector = (
    hyde_embeddings
    .embed_query(
        query
    )
)


### Learning: hyde_documents = (

**What you'll learn:** Execute the next step in the retrieval pipeline and observe the output.

**What this cell does:** Executes retrieval/generation for: hyde_documents = (.

**Watch for:** Relate this step to find → order → trim in the RAG pipeline.



In [ ]:
hyde_documents = (
    vector_store
    .similarity_search_by_vector(
        embedding=hyde_vector,
        k=4
    )
)

### Learning: PRINT COMPARISON

**What you'll learn:** Attach section/page metadata used later for filters.

**What this cell does:** Runs `PRINT COMPARISON` and prints intermediate results you can inspect.

**Watch for:** Good metadata is what makes pre-filtering possible.



In [ ]:
# ============================================================
# 23. PRINT COMPARISON
# ============================================================

print(
    "\nNORMAL DENSE RETRIEVAL"
)

print(
    "=" * 80
)

for i, doc in enumerate(
    normal_documents,
    start=1
):

    print(
        i,
        "| Page:",
        doc.metadata.get(
            "paper_page"
        ),
        "| Section:",
        doc.metadata.get(
            "section"
        ),
        "|",
        doc.page_content[:200]
        .replace(
            "\n",
            " "
        )
    )


print(
    "\nHYDE RETRIEVAL"
)

print(
    "=" * 80
)

for i, doc in enumerate(
    hyde_documents,
    start=1
):

    print(
        i,
        "| Page:",
        doc.metadata.get(
            "paper_page"
        ),
        "| Section:",
        doc.metadata.get(
            "section"
        ),
        "|",
        doc.page_content[:200]
        .replace(
            "\n",
            " "
        )
    )



NORMAL DENSE RETRIEVAL
1 | Page: 3 | Section: introduction | continue to improve the safety of those models, paving the way for more responsible development of LLMs. We also share novel observations we made during the development ofLlama 2 and Llama 2-Chat, suc
2 | Page: 3 | Section: introduction | the community to advance AI alignment research. In this work, we develop and release Llama 2, a family of pretrained and fine-tuned LLMs,Llama 2 and Llama 2-Chat, at scales up to 70B parameters. On th
3 | Page: 4 | Section: introduction | Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed- source models. Human raters judged model generations for safety violations across ~2,000 adversarial
4 | Page: 4 | Section: introduction | 1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also increased the size of the pretraining corpus by 40%, doubled the context length of the model, and ado

HYDE RETRIEVAL
1 | 

### Learning: FINAL FLOW

**What you'll learn:** Retrieve by semantic nearest-neighbors.

**What this cell does:** Runs `FINAL FLOW` and prints intermediate results you can inspect.

**Watch for:** Dense can miss exact codes/IDs — compare with BM25 later.



In [ ]:
# ============================================================
# 24. FINAL FLOW
# ============================================================

"""
NORMAL DENSE RETRIEVAL

User Query
    ↓
Query Embedding
    ↓
Vector Search
    ↓
Relevant Documents


HYDE RETRIEVAL

User Query
    ↓
LLM generates hypothetical document
    ↓
Hypothetical document embedding
    ↓
Vector Search
    ↓
Relevant REAL documents
"""


print(
    "\nPractical completed successfully."
)

### Learning: hypothetical_document = hyde_embeddings.llm_chain.invoke(

**What you'll learn:** Create vectors that put similar meaning nearby.

**What this cell does:** Executes retrieval/generation for: hypothetical_document = hyde_embeddings.llm_chain.invoke(.

**Watch for:** Note dimension size; it must match the index.



In [ ]:
hypothetical_document = hyde_embeddings.llm_chain.invoke(
    {"query": query}
)

hyde_vector = hyde_embeddings.embed_query(query)

hyde_documents = vector_store.similarity_search_by_vector(
    embedding=hyde_vector,
    k=4
)